# Bank Loan Analysis
#### Phase 3: Preprocessing + Modeling + Valuation 
This Notebook builds and evaluates several models to help detect a loan that has high risk of defaulting due to applicant and loan features. The idea is to reduce risk but optimize approvals by keeping acceptance high.

In This Notebook: 
- Define target variable and feature variables.
- Create a shared train/validation/test split for fair model comparison.
- create preprocessing pipelines 
- Build, evaluate and compare models to choose the best one
- Tune chosen model structure for efficiency of the final model 

Decision Class [default_flag]

- 0 → Loan Repaid (negative)

- 1 → Loan Defaulted (positive)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

# Visualization (optional but handy)
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
import xgboost as xgb
import joblib
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report


# Notebook 3 configuration
NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parents[1]

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

ENGINEERED_INPUT_PATH = PROCESSED_DIR / "bank_loan_data_engineered.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("ENGINEERED_INPUT_PATH:", ENGINEERED_INPUT_PATH, "exists?", ENGINEERED_INPUT_PATH.exists())

# Load engineered data for modeling
df = pd.read_csv(ENGINEERED_INPUT_PATH)
df.head()


PROJECT_ROOT: /Users/macbook/projects/data-science/bank-loan-risk
ENGINEERED_INPUT_PATH: /Users/macbook/projects/data-science/bank-loan-risk/data/processed/bank_loan_data_engineered.csv exists? True


,age,gender,education,income,employment_experience_length,home_ownership,loan_amount,loan_purpose,loan_interest_rate,loan_percent_to_income,...,age_outlier_z,income_capped,loan_amount_capped,loan_percent_to_income_capped,default_flag,age_bucket,credit_score_band,loan_percent_to_income_band,income_band,loan_amount_band
0,22.0,female,Master,71948.0,0,RENT,35000.0,PERSONAL,16.02,0.49,...,False,71948.0,28393.06,0.40,0,18_25,Poor,High / Risky,"Middle (60k–119,999)","Lower-mid (25,001–100k)"
1,21.0,female,High School,12282.0,0,OWN,1000.0,EDUCATION,11.14,0.08,...,False,12282.0,1000.00,0.08,1,18_25,Poor,Low / Strong,Low (<30k),Micro / very small (0–5k)
2,25.0,female,High School,12438.0,3,MORTGAGE,5500.0,MEDICAL,12.87,0.44,...,False,12438.0,5500.00,0.40,0,18_25,Fair,High / Risky,Low (<30k),"Small (5,001–25k)"
3,23.0,female,Bachelor,79753.0,0,RENT,35000.0,MEDICAL,15.23,0.44,...,False,79753.0,28393.06,0.40,0,18_25,Good,High / Risky,"Middle (60k–119,999)","Lower-mid (25,001–100k)"
4,24.0,male,Master,66135.0,1,RENT,35000.0,MEDICAL,14.27,0.53,...,False,66135.0,28393.06,0.40,0,18_25,Fair,Excessive,"Middle (60k–119,999)","Lower-mid (25,001–100k)"


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 44992 entries, 0 to 44991
Data columns (total 37 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   age                                 44992 non-null  float64
 1   gender                              44992 non-null  object 
 2   education                           44992 non-null  object 
 3   income                              44992 non-null  float64
 4   employment_experience_length        44992 non-null  int64  
 5   home_ownership                      44992 non-null  object 
 6   loan_amount                         44992 non-null  float64
 7   loan_purpose                        44992 non-null  object 
 8   loan_interest_rate                  44992 non-null  float64
 9   loan_percent_to_income              44992 non-null  float64
 10  credit_history_length               44992 non-null  float64
 11  credit_score                        44992

In [3]:
y = df['default_flag']  

num_cols = [
    "income_capped",
    "loan_amount_capped",
    "loan_interest_rate",
    "loan_percent_to_income_capped",
    "credit_history_length",
    "employment_experience_length"]

cat_cols = [
    "loan_purpose", 
    "gender",
    "education",
    "home_ownership",
    "previous_loan_default",
    "age_bucket",
    "credit_score_band",]


X = df[num_cols + cat_cols]


In [9]:

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, random_state=21, stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, random_state=21, stratify=y_temp
)

print(X_train.shape, X_val.shape, y_train.shape, y_val.shape)

(26994, 13) (8999, 13) (26994,) (8999,)


In [10]:
df['default_flag'].value_counts(normalize=True) *100


default_flag
1    77.773826
0    22.226174
Name: proportion, dtype: float64

In [56]:
# Create Numerical and Categorical Transformers

numerical_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(
    handle_unknown="ignore",
    sparse_output=False,
)

# Combine transformers into a preprocessor
preprocessor = ColumnTransformer(
    transformers=[
        ("numbers", numerical_transformer, num_cols),
        ("categories", categorical_transformer, cat_cols),
    ],
)

# Build Logistic Regression Model 
logreg = LogisticRegression(
    penalty="l2",
    C=1.0,
    solver="lbfgs",
    max_iter=1000,
    class_weight="balanced",
)

# Create the pipeline
logregpipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", logreg),
    ],
)


In [57]:

# Fit Model
logregpipeline.fit(X_train, y_train)

# Run Predictions on Validation Set (probabilities and classes)
y_val_proba = logregpipeline.predict_proba(X_val)[:, 1]
y_val_pred = logregpipeline.predict(X_val)

# Evaluate Model
val_roc_auc = roc_auc_score(y_val, y_val_proba)
print("Validation ROC AUC:", val_roc_auc)

print("Validation Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred))

print("Validation Classification Report:")
print(classification_report(y_val, y_val_pred, digits=3))

Validation ROC AUC: 0.9573031147306758
Validation Confusion Matrix:
[[1853  147]
 [1119 5880]]
Validation Classification Report:
              precision    recall  f1-score   support

           0      0.623     0.926     0.745      2000
           1      0.976     0.840     0.903      6999

    accuracy                          0.859      8999
   macro avg      0.800     0.883     0.824      8999
weighted avg      0.897     0.859     0.868      8999



/Users/macbook/projects/data-science/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


## Model 1: Logistic Regression with Class Weights Balanced Metrics Evaluataion 

#### ROC-AUC 

- score of 0.96 (Excellent Discrimination)
    - Our Model very relaibly ranks truly risky borrowers above the safe ones, which is crucial if we are going to edit probability threaholds for approvals
#### Recall for Class 1 (Positive: defaulted loans)
- Score of 0.84 
- About 84% of Defaulters are caught. The bank misses about 16% of risky loans at this operating point

#### Precision for Claqss 1(Positive: Defaulted Loans)
- Score of 0.98 
- When the model predicts "default", it is almost always right. 

#### F-1 Score for Class 1
- Score of 0.90
- Balanced strength in both catching defaulters and being correct when flagging them



In [18]:
# Project Goal 1: Identify and better understand which applicant and loan features are most strongly associated with a loan being repaid versus defaulted

# Extract feature names and get coefficients from the logistic regression model
feature_names = logregpipeline.named_steps["preprocessor"].get_feature_names_out()
coefs = logregpipeline.named_steps["classifier"].coef_[0]
odds = np.exp(coefs)

# Create a DataFrame to display feature names, coefficients, and odds ratios
coef_table = pd.DataFrame({
    "feature": feature_names,
    "coef": coefs,
    "odds_ratio": odds
}).sort_values("odds_ratio", ascending=False)

coef_table.head(50)

,feature,coef,odds_ratio
24,categories__previous_loan_default_Yes,5.101284,164.232584
21,categories__home_ownership_OWN,1.656562,5.241262
11,categories__loan_purpose_VENTURE,0.994322,2.702890
1,numbers__loan_amount_capped,0.794788,2.213971
32,categories__credit_score_band_Very good,0.775627,2.171953
30,categories__credit_score_band_Good,0.723796,2.062247
12,categories__gender_female,0.652696,1.920712
13,categories__gender_male,0.648689,1.913031
7,categories__loan_purpose_EDUCATION,0.549906,1.733091
27,categories__age_bucket_36_50,0.423900,1.527910


### We inspected logistic regression coefficients as odds ratios to understand which borrower and loan attributes drive default risk

In [36]:
# Feature Influence Breakdown 

# 2) Top 5 risk-increasing features (largest odds ratios)
top_5_increase = (
    coef_table
    .sort_values("odds_ratio", ascending=False)
    .head(5)
    .copy()
)

# 3) Top 5 risk-reducing features (smallest odds ratios)
top_5_decrease = (
    coef_table
    .sort_values("odds_ratio", ascending=True)
    .head(5)
    .copy()
)

# 4) Optional: round for cleaner notebook display
for df in [top_5_increase, top_5_decrease]:
    df["coef"] = df["coef"].round(6)
    df["odds_ratio"] = df["odds_ratio"].round(6)


In [31]:
# 2) Build natural-language interpretations WITH odds / percentages
def make_interpretation(row):
    feature = row["feature"]
    or_val = row["odds_ratio"]

    if or_val > 1:
        if feature.startswith("numbers__"):
            return f"A higher value for this feature is associated with about {or_val:.2f}x higher odds that the loan will default, holding other factors constant."
        else:
            return f"Being in this category is associated with about {or_val:.2f}x higher odds that the loan will default compared with the baseline group."
    else:
        pct_lower = (1 - or_val) * 100

        if feature.startswith("numbers__"):
            return f"A higher value for this feature is associated with about {pct_lower:.1f}% lower odds that the loan will default, holding other factors constant."
        else:
            return f"Being in this category is associated with about {pct_lower:.1f}% lower odds that the loan will default compared with the baseline group."

coef_table["interpretation"] = coef_table.apply(make_interpretation, axis=1)

# 3) Top 5 increasing and top 5 reducing features (by odds_ratio)
top_5_increase_table = (
    coef_table
    .sort_values("odds_ratio", ascending=False)
    .head(5)[["feature", "interpretation"]]
    .rename(columns={"feature": "feature name"})
)

top_5_decrease_table = (
    coef_table
    .sort_values("odds_ratio", ascending=True)
    .head(5)[["feature", "interpretation"]]
    .rename(columns={"feature": "feature name"})
)

In [32]:
# Make columns wider and allow long strings
pd.set_option("display.max_colwidth", None)   # no truncation of cell text
pd.set_option("display.width", 200)          # overall table width in characters

print("### Top 5 features increasing odds of default")
display(top_5_increase_table)

print("\n### Top 5 features reducing odds of default")
display(top_5_decrease_table)



### Top 5 features increasing odds of default


,feature name,interpretation
24,categories__previous_loan_default_Yes,Being in this category is associated with about 164.23x higher odds that the loan will default compared with the baseline group.
21,categories__home_ownership_OWN,Being in this category is associated with about 5.24x higher odds that the loan will default compared with the baseline group.
11,categories__loan_purpose_VENTURE,Being in this category is associated with about 2.70x higher odds that the loan will default compared with the baseline group.
1,numbers__loan_amount_capped,"A higher value for this feature is associated with about 2.21x higher odds that the loan will default, holding other factors constant."
32,categories__credit_score_band_Very good,Being in this category is associated with about 2.17x higher odds that the loan will default compared with the baseline group.



### Top 5 features reducing odds of default


,feature name,interpretation
23,categories__previous_loan_default_No,Being in this category is associated with about 97.8% lower odds that the loan will default compared with the baseline group.
3,numbers__loan_percent_to_income_capped,"A higher value for this feature is associated with about 76.8% lower odds that the loan will default, holding other factors constant."
2,numbers__loan_interest_rate,"A higher value for this feature is associated with about 62.4% lower odds that the loan will default, holding other factors constant."
31,categories__credit_score_band_Poor,Being in this category is associated with about 41.5% lower odds that the loan will default compared with the baseline group.
22,categories__home_ownership_RENT,Being in this category is associated with about 37.6% lower odds that the loan will default compared with the baseline group.


In [34]:
# Inspect different thresholds and choose the best one that fits business needs
from sklearn.metrics import precision_recall_curve, f1_score

precision, recall, thresholds = precision_recall_curve(y_val, y_val_proba)

thr_df = pd.DataFrame({
    "threshold": np.append(thresholds, 1.0),  # align length with precision/recall
    "precision": precision,
    "recall": recall,
})

# For each threshold, compute F1 on the fly
def f1_at_threshold(t):
    preds = (y_val_proba >= t).astype(int)
    return f1_score(y_val, preds)

thr_df["f1"] = thr_df["threshold"].apply(f1_at_threshold)
thr_df.sort_values("threshold", inplace=True)
thr_df.head(), thr_df.tail()

(   threshold  precision  recall        f1
 0   0.000503   0.777753     1.0  0.874984
 1   0.000709   0.777840     1.0  0.875039
 2   0.000759   0.777926     1.0  0.875094
 3   0.001165   0.778012     1.0  0.875148
 4   0.001205   0.778099     1.0  0.875203,
       threshold  precision    recall        f1
 8995   0.999998        1.0  0.000572  0.001142
 8996   0.999998        1.0  0.000429  0.000857
 8997   0.999998        1.0  0.000286  0.000571
 8998   0.999999        1.0  0.000143  0.000286
 8999   1.000000        1.0  0.000000  0.000000)

In [ ]:
# Helper: pick best threshold for a minimum recall
def pick_threshold_for_recall(thr_df, target_recall):
    # keep only rows meeting or exceeding target recall
    candidates = thr_df[thr_df["recall"] >= target_recall].copy()
    if candidates.empty:
        raise ValueError(f"No thresholds achieve recall >= {target_recall:.2f}")
    # among those, choose the row with highest precision (or F1 if you prefer)
    best = candidates.sort_values("precision", ascending=False).iloc[0]
    return {
        "target_recall": target_recall,
        "threshold": float(best["threshold"]),
        "precision": float(best["precision"]),
        "recall": float(best["recall"]),
        "f1": float(best["f1"]),
    }

best_07 = pick_threshold_for_recall(thr_df, 0.7)
best_08 = pick_threshold_for_recall(thr_df, 0.8)

best_07, best_08

({'target_recall': 0.7,
  'threshold': 0.8647404267197636,
  'precision': 0.9983729916615822,
  'recall': 0.7013859122731819,
  'f1': 0.8239342061094327},
 {'target_recall': 0.8,
  'threshold': 0.6168805782727111,
  'precision': 0.9853950378321309,
  'recall': 0.800114302043149,
  'f1': 0.8831414603374862})

In [38]:
# Evaluate the model with the chosen threshold
target_recall = 0.8
candidates = thr_df[thr_df["recall"] >= target_recall]

best = candidates.sort_values("precision", ascending=False).iloc[0]
best_threshold = float(best["threshold"])
best
t = best_threshold  # or a manual choice like 0.3

y_val_custom = (y_val_proba >= t).astype(int)

print("Custom threshold:", t)
print("ROC AUC (unchanged):", roc_auc_score(y_val, y_val_proba))
print("Confusion matrix:\n", confusion_matrix(y_val, y_val_custom))
print("\nClassification report:\n", classification_report(y_val, y_val_custom, digits=3))

Custom threshold: 0.6168805782727111
ROC AUC (unchanged): 0.9573031147306758
Confusion matrix:
 [[1917   83]
 [1399 5600]]

Classification report:
               precision    recall  f1-score   support

           0      0.578     0.959     0.721      2000
           1      0.985     0.800     0.883      6999

    accuracy                          0.835      8999
   macro avg      0.782     0.879     0.802      8999
weighted avg      0.895     0.835     0.847      8999



Threshold comparison on validation set

| Threshold | Precision (default) | Recall (default) | Predicted default (count, %) | Predicted repaid (count, %) |
|----------|----------------------|------------------|------------------------------|-----------------------------|
| 0.50     | 0.976               | 0.840           | 6027 (67.0%)                 | 2972 (33.0%)                |
| 0.617    | 0.985               | 0.800           | 5683 (63.1%)                 | 3316 (36.8%)                |

### Threshold selection and business interpretation
AGE Analytics requires the model to correctly identify at least 80% of potential defaulters, since approving a risky loan is substantially more costly than declining a borrower who is statistically likely to repay. To meet this risk appetite, we tune the decision threshold on the validation set using the predicted default probabilities from the logistic regression model.

The selected operating point is a probability threshold of 0.617: any applicant with a predicted default probability greater than or equal to 61.7% is classified as “default.” At this threshold, the model achieves roughly 80% recall for defaulted loans, meaning it successfully flags about four out of five loans that will go on to default. In exchange, we accept a small increase in false alarms: precision for the default class decreases from approximately 99.8% at a stricter threshold to about 98.5% at 0.617, which implies that around 1.5% of loans flagged as “default” would, in fact, repay.

In practice, lenders typically set such thresholds using explicit cost and portfolio targets—for example, profit per performing loan versus loss given default—rather than a fixed industry‑standard recall value. In this demonstration, we adopt a minimum recall of 0.8 for defaulters as a clear, conservative policy target that prioritizes risk containment while still keeping model‑recommended approvals relatively high.




In [43]:
t_low = 0.2
t_high = 0.4

def decision_from_prob(p):
    if p < t_low:
        return "approve"
    elif p < t_high:
        return "review"
    else:
        return "reject"

val_df = pd.DataFrame({
    "y_true": y_val,
    "p_default": y_val_proba,
})
val_df["decision"] = val_df["p_default"].apply(decision_from_prob)

# Share of loans in each band
band_share = val_df["decision"].value_counts(normalize=True)

# Default rate inside each band
band_stats = val_df.groupby("decision")["y_true"].agg(
    n_loans="count",
    default_rate="mean",
)

band_share, band_stats

(decision
 reject     0.707523
 approve    0.205023
 review     0.087454
 Name: proportion, dtype: float64,
           n_loans  default_rate
 decision                       
 approve      1845      0.200542
 reject       6367      0.965447
 review        787      0.612452)

## Validation band results

Using `t_low = 0.2` and `t_high = 0.4`, the validation-set policy splits loans into three decision bands.[page:1]

### Share of loans in each band

- **Reject:** 70.8% of loans.[page:1]
- **Approve:** 20.5% of loans.[page:1]
- **Review:** 8.7% of loans.[page:1]

### Default rate within each band

Remember: `1 = default` and `0 = repaid`.[page:1]

- **Approve:** 20.05% default rate.[page:1]
- **Review:** 61.25% default rate.[page:1]
- **Reject:** 96.54% default rate.[page:1]

### Interpretation

With `t_low = 0.2` and `t_high = 0.4`, the approve band is still fairly risky: about 1 in 5 approved loans default.[page:1] The review band is very risky, with roughly 3 in 5 loans defaulting.[page:1] The reject band captures extremely high-risk loans, with a default rate above 96%.[page:1]

These results suggest two things.[page:1]

- The dataset is highly skewed toward default, so defaults dominate most score ranges.[page:1]
- The current cutoffs are quite conservative: the policy rejects most applications, and even the lowest-risk group still has meaningful default risk.[page:1]

In [58]:
# Refit the pipeline on train + validation
X_train_full = pd.concat([X_train, X_val])
y_train_full = pd.concat([y_train, y_val])

logregpipeline.fit(X_train_full, y_train_full)

/Users/macbook/projects/data-science/.venv/lib/python3.13/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('classifier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('numbers', ...), ('categories', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different

In [46]:

# Use the same threshold you selected on validation
t = 0.6168805782727111  # or float(best08["threshold"]) if best08 is still in memory

# Predict probabilities and classes on test
y_test_proba = logregpipeline.predict_proba(X_test)[:, 1]
y_test_custom = (y_test_proba >= t).astype(int)

print("Test threshold:", t)
print("Test ROC AUC:", roc_auc_score(y_test, y_test_proba))
print("\nTest confusion matrix:\n", confusion_matrix(y_test, y_test_custom))
print("\nTest classification report:\n", classification_report(y_test, y_test_custom, digits=3))

Test threshold: 0.6168805782727111
Test ROC AUC: 0.9502571081583084

Test confusion matrix:
 [[1902   98]
 [1437 5562]]

Test classification report:
               precision    recall  f1-score   support

           0      0.570     0.951     0.712      2000
           1      0.983     0.795     0.879      6999

    accuracy                          0.829      8999
   macro avg      0.776     0.873     0.796      8999
weighted avg      0.891     0.829     0.842      8999



In [55]:

test_df = pd.DataFrame({
    "y_true": y_test,
    "p_default": y_test_proba,
})
test_df["decision"] = test_df["p_default"].apply(decision_from_prob)

print("Test decision mix:")
print(test_df["decision"].value_counts(normalize=True))

print("\nDefault rate by decision band (test):")
band_summary = test_df.groupby("decision")["y_true"].agg(
    n_loans="count",
    default_rate="mean",
)
print(band_summary)

Test decision mix:
decision
reject     0.709079
approve    0.223469
review     0.067452
Name: proportion, dtype: float64

Default rate by decision band (test):
          n_loans  default_rate
decision                       
approve      2011      0.247141
reject       6381      0.956903
review        607      0.652389


### Final Threshold and Policy Selection

We experimented with slightly lowering the recall target to 0.75 and increasing the lower approval cutoff \(t_\text{low}\) above 0.20, but the resulting decision mix and default rates on the test set were not materially better. As a result, we retain the original configuration with a recall target of approximately 0.80, a main decision threshold of about 0.617, and bands defined by \(t_\text{low} = 0.20\) and \(t_\text{high} = 0.40\).[file:88]

Under this final policy on the held‑out test set of 8,999 loans, the model:
- **Approves** about 19.8% of applications with a default rate of roughly 20.9%.
- Routes **9.3%** of loans to **manual review** with a default rate of about 62.2%.
- **Rejects** 70.9% of applicants whose default rate is approximately 95.7%.

This pattern shows that the tuned logistic regression model successfully concentrates the vast majority of credit risk into the reject segment while maintaining a conservative approval strategy. However, it is important to note that these figures are not representative of a typical retail lending portfolio: in this engineered dataset, approximately 77.8% of loans are labeled as defaulted and only 22.2% as repaid, which is the opposite of real‑world portfolios where defaults are usually the minority class. The unusually high share of defaulted loans inflates observed default rates in all bands and makes the approval policy appear more extreme than would be expected on an actual bank dataset. In practice, lenders would calibrate thresholds and bands using data with realistic default rates and explicit economic constraints (e.g., loss given default, profitability targets, and capital requirements).